In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
cut = 0.01
folder = 'TCJA_Ext_Plus_Spending_Cut_'+str(cut)
reform_dir = os.path.join(CUR_DIR, folder, "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94,-4.94
1,IIT: Pct Change due to behavior,1.14,1.12,1.12,1.11,1.11,1.11,1.10,1.10,1.10,1.10,1.11,1.12
2,IIT: Pct Change due to macro,-0.02,-0.03,-0.03,-0.03,-0.02,-0.02,-0.02,-0.02,-0.02,-0.02,-0.02,0.16
3,IIT: Overall Pct Change in taxes,-3.82,-3.84,-3.84,-3.85,-3.85,-3.85,-3.85,-3.85,-3.85,-3.86,-3.85,-3.67
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.90,0.94,1.01,1.05,1.09,1.12,1.15,1.18,1.20,1.22,1.09,1.84
6,CIT: Pct Change due to macro,0.48,0.37,0.26,0.17,0.09,0.02,-0.03,-0.08,-0.13,-0.16,0.10,-0.87
7,CIT: Overall Pct Change in taxes,1.38,1.31,1.26,1.22,1.18,1.15,1.12,1.10,1.07,1.06,1.18,0.96
8,All: Pct Change due to tax rates,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65,-4.65
9,All: Pct Change due to behavior,1.13,1.11,1.11,1.11,1.11,1.11,1.11,1.11,1.11,1.11,1.11,1.16


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2025-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]

# Taken from CBO June 2024 Budgdet Outlook, 2025-2034
base_spending = np.array([6.975, 7.244, 7.512, 7.886, 8.082, 8.547, 8.944, 9.387, 9.998, 10.320])
reform_scale = np.array([1-cut, (1-cut)**2, (1-cut)**3, (1-cut)**4, (1-cut)**4, (1-cut)**4, (1-cut)**4, (1-cut)**4, (1-cut)**4, (1-cut)**4])
reform_spending = np.multiply(base_spending, reform_scale)
spending_change = reform_spending - base_spending
df_levels.loc[12] = ['Total Spending Change'] + list(spending_change) + [sum(spending_change)] 

df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.23,-0.25,-0.27,-0.28,-0.29,-0.30,-0.31,-0.32,-0.33,-0.35,-2.92
9,Rev Change Due to Behavior,0.06,0.06,0.06,0.07,0.07,0.07,0.07,0.08,0.08,0.08,0.70
10,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.01
11,Total Revenue Change,-0.18,-0.19,-0.20,-0.21,-0.22,-0.23,-0.24,-0.25,-0.26,-0.27,-2.24
12,Total Spending Change,-0.07,-0.14,-0.22,-0.31,-0.32,-0.34,-0.35,-0.37,-0.39,-0.41,-2.93
